<center><h2> Generating LSST multi-band postage stamps

In this notebook, we generate samples of various sizes for the time-delay measurement project. The end result of this notebook is a csv generated with the following properties (at minimum):
* microlensing parameters
    * convergence: $\kappa$
    * stellar convergence: $\kappa_*$
    * shear: $\gamma$
* black hole accretion and transfer function parameters required by AMOEBA
    * black hole mass
    * eddington rate
    * black hole accretion disk inclination angle
* variability parameters required for a damped random walk generated using a bending power-law with fixed lower slope of 0 and higher slope of 2
    * $SF_\infty$ = $\sqrt2\sigma$ where $\sigma$ is the standard deviation of light curve variation from the mean
    * $\tau_{\rm DRW}$ where the breakpoint frequency = $1/\sqrt(2\pi\tau_{\rm DRW})$
* lensed magnitudes of 2/3/4 images
* arrival times of 2/3/4 images

All of this information is stored as a metadata in the final hdf5 file with time series postage stamps.

### Import required pacakges

In [1]:
### Cosmology and astropy packages
from astropy.cosmology import FlatLambdaCDM
from astropy.units import Quantity
import astropy.coordinates as coord
import astropy.units as u

### Arrays, tables, plots
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

### SLSim functions
import slsim.Sources as sources
import slsim.Deflectors as deflectors
import slsim.Pipelines as pipelines
from slsim.Sources.SourceCatalogues.QuasarCatalog.quasar_pop import QuasarRate
from slsim.Lenses.lens_pop import LensPop
from slsim.ImageSimulation.image_simulation import (
    point_source_coordinate_properties,
    lens_image_series_precomputed_mags,
)


from slsim.Util.param_util import ellipticity2phi_q

# from slsim.Util.distribution_plot_utils import make_contour
from slsim.LsstSciencePipeline.rubin_sim_pipeline import get_rubin_cadence
from lenstronomy.Util.data_util import bkg_noise

### Readin, readout, paths
from contextlib import redirect_stdout
import io
from tqdm import tqdm
from tqdm import tqdm

### recompile packages after each edit
%load_ext autoreload
%autoreload 2

/global/common/software/m1727/vpadma/slsim_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Set up SLSim to generate populations

In [2]:
# define a cosmology
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

# define a sky area
galaxy_sky_area = Quantity(
    value=10, unit="deg2"
)  # this is the sky area over which galaxies are sampled
quasar_sky_area = Quantity(value=10, unit="deg2")

# this is the sky area over which lensed quasars are sampled
full_sky_area = Quantity(value=5000, unit="deg2")

# define limits in the intrinsic deflector and source population (in addition
# to the skypy config
# file)
kwargs_deflector_cut = {"band": "i", "band_max": 28, "z_min": 0.01, "z_max": 2.5}
kwargs_source_cut = {"band": "i", "band_max": 26, "z_min": 0.001, "z_max": 6.0}

In [3]:
# generate galaxy population using skypy pipeline.
galaxy_simulation_pipeline = pipelines.SkyPyPipeline(
    skypy_config=None,
    sky_area=galaxy_sky_area,
    filters=["u", "g", "r", "i", "z", "y"],
    cosmo=cosmo,
    z_min=0,
)

### Generate lens galaxy population

In [4]:
lens_galaxies_ell = deflectors.EllipticalLensGalaxies(
    galaxy_list=galaxy_simulation_pipeline.red_galaxies,
    kwargs_cut=kwargs_deflector_cut,
    kwargs_mass2light={},
    cosmo=cosmo,
    sky_area=galaxy_sky_area,
    gamma_pl=dict(mean=2.0, std_dev=0.16),
)

/global/u1/v/vpadma/postage_stamp_generation/src/slsim/slsim/Deflectors/DeflectorPopulation/elliptical_lens_galaxies.py:48: UserWarning: Angular size is converted to arcsec because provided input_catalog_type is skypy. If this is not correct, please refer to the documentation of the class you are using
  galaxy_list = param_util.catalog_with_angular_size_in_arcsec(


In [5]:
# Initiate QuasarRate class to generate quasar sample.
quasar_class = QuasarRate(
    cosmo=cosmo,
    sky_area=quasar_sky_area,
    noise=True,
    redshifts=np.linspace(0.001, 6.00, 100),  # these redshifts are provided
    # to match general slsim redshift range in skypy pipeline.
)
# quasar sample with host galaxy
quasar_source_plus_galaxy = quasar_class.quasar_sample(
    m_min=15, m_max=28, host_galaxy=True
)

Matching quasars with host galaxies: 100%|█████████████████████████████████████████████████████████████████████| 45466/45466 [00:21<00:00, 2072.86it/s]


In [6]:
# Prepare dictionary of agn variability kwargs
length_of_light_curve = 3850
MACLEOD2010_MEANS = np.array(
    [8.53308079, -23.48721021, -0.51665998, 2.28708691, 2.11640976]
)
MACLEOD2010_COV = np.array(
    [
        [0.27862905, -0.29501766, 0.00675703, 0.04606804, -0.00665875],
        [-0.29501766, 2.06855169, 0.19690851, 0.0244139, -0.29913764],
        [0.00675703, 0.19690851, 0.02785685, 0.01083628, -0.02216221],
        [0.04606804, 0.0244139, 0.01083628, 0.05636087, -0.02716507],
        [-0.00665875, -0.29913764, -0.02216221, -0.02716507, 0.3077278],
    ]
)
#############################################################################


# Prepare dictionary of agn variability kwargs
# Note: the means array and covariance matrix should be defined in following order and units:
# log(BH_mass/Msun), M_i, log(SFi_inf/mag), log(tau/days), zsrc
variable_agn_kwarg_dict = {
    "multivariate_gaussian_means": MACLEOD2010_MEANS,
    "multivariate_gaussian_covs": MACLEOD2010_COV,
    "known_band": "lsst2016-i",
}
# variable_agn_kwarg_dict = {
#     "length_of_light_curve": length_of_light_curve,
#     "time_resolution": 1,
#     # "log_breakpoint_frequency": 1 / 20,
#     # "low_frequency_slope": 1,
#     # "high_frequency_slope": 3,
#     # "standard_deviation": 0.9,
# }
# variable_agn_kwarg_dict = {}
kwargs_quasar = {
    "variability_model": "light_curve",
    "kwargs_variability": {"agn_lightcurve", "u", "g", "r", "i", "z", "y"},
    "agn_driving_variability_model": "bending_power_law_from_distribution",
    "agn_driving_kwargs_variability": variable_agn_kwarg_dict,
    "lightcurve_time": np.linspace(0, length_of_light_curve, length_of_light_curve),
    "corona_height": 10,
    "r_resolution": 500,
}
# Initiate source population class.
source_quasar_plus_galaxies = sources.PointPlusExtendedSources(
    point_plus_extended_sources_list=quasar_source_plus_galaxy,
    cosmo=cosmo,
    sky_area=quasar_sky_area,
    kwargs_cut=kwargs_source_cut,
    list_type="astropy_table",
    catalog_type="skypy",
    point_source_type="quasar",
    extended_source_type="single_sersic",
    point_source_kwargs=kwargs_quasar,
)

In [7]:
# Initiate LensPop class to generate lensed quasar pop.
quasar_lens_pop_ell = LensPop(
    deflector_population=lens_galaxies_ell,
    source_population=source_quasar_plus_galaxies,
    cosmo=cosmo,
    sky_area=full_sky_area,
)
image_sep = 0.5
mag_lim = 24
kwargs_lens_cuts = {
    "min_image_separation": image_sep,
    "max_image_separation": 10,
    "second_brightest_image_cut": {"i": mag_lim},
}
# drawing population
# the key difference in lens population drawing time is whether you ask for magnitude cuts or not I think?
quasar_lens_population = []
for i in tqdm(range(2)):
    qlp5000 = quasar_lens_pop_ell.draw_population(
        speed_factor=1000, kwargs_lens_cuts=kwargs_lens_cuts
    )
    quasar_lens_population.extend(qlp5000)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [10:40<00:00, 320.23s/it]


### Make the dataframe for the population

In [8]:
import warnings
warnings.filterwarnings('ignore')

In [9]:
f = io.StringIO()
full_pop_df = pd.DataFrame()
with redirect_stdout(f):
    for i, lens_obj in tqdm(enumerate(quasar_lens_population)):
        full_pop_df = lens_obj.lens_to_dataframe(index=i, df=full_pop_df)
        image2mag = full_pop_df.loc[i, "point_source_light_i_magnitude_1"]
        try:
            image3mag = full_pop_df.loc[i, "point_source_light_i_magnitude_2"]
        except KeyError:
            image3mag = 0
        second_or_third_mag = (
            image3mag if not (np.isnan(image3mag) or image3mag == 0) else image2mag)
        
        full_pop_df.loc[i, "i2"] = image2mag
        full_pop_df.loc[i, "i3"] = second_or_third_mag
        (
            full_pop_df.loc[i, "deflector_mass_phi"],
            full_pop_df.loc[i, "deflector_mass_q"],
        ) = ellipticity2phi_q(
            full_pop_df.loc[i, "deflector_mass_e1"],
            full_pop_df.loc[i, "deflector_mass_e2"],
        )
        (
            full_pop_df.loc[i, "deflector_light_phi"],
            full_pop_df.loc[i, "deflector_light_q"],
        ) = ellipticity2phi_q(
            full_pop_df.loc[i, "deflector_light_i_e1"],
            full_pop_df.loc[i, "deflector_light_i_e2"],
        )
        full_pop_df.loc[i, "deflector_stellar_mass"] = lens_obj.deflector_stellar_mass()
        full_pop_df.loc[i, "lens_obj"] = lens_obj
        for band in list("ugrizy"):
            abs_mag = quasar_class.convert_magnitude(
                full_pop_df.loc[i, f"ps_{band}_mag_true"],
                full_pop_df.loc[i, "point_source_redshift"],
                conversion="apparent_to_absolute",
            )
            full_pop_df.loc[i, f"M_{band}"] = abs_mag
            if np.isnan(abs_mag):
                if band == "y":
                    full_pop_df.drop(index=i, inplace=True)
                    quasar_lens_population.pop(i)

1502it [54:03,  2.16s/it]


In [10]:
full_pop_df

,ID,deflector_mass_theta_E,deflector_mass_gamma,deflector_mass_center_x,deflector_mass_center_y,deflector_mass_e1,deflector_mass_e2,deflector_mass_gamma1,deflector_mass_gamma2,deflector_mass_ra_0,...,micro_kappa_star_2,micro_kappa_star_3,micro_kappa_tot_2,micro_kappa_tot_3,micro_shear_2,micro_shear_3,micro_shear_angle_2,micro_shear_angle_3,point_source_arrival_time_2,point_source_arrival_time_3
0,GAL-QSO-LENS_0.0342_-0.0259,0.975289,2.286384,0.034182,-0.025875,0.052787,-0.113628,-0.051010,0.037388,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GAL-QSO-LENS_-0.0146_0.0435,0.461350,2.156316,-0.014595,0.043540,0.002516,-0.056561,0.037295,0.002854,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,GAL-QSO-LENS_0.0477_0.0022,0.760769,2.052386,0.047680,0.002182,-0.000588,0.227549,-0.047629,-0.003821,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,GAL-QSO-LENS_-0.0503_-0.0059,0.806669,2.029320,-0.050346,-0.005874,-0.074383,-0.034664,0.000000,0.000000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,GAL-QSO-LENS_-0.0350_-0.0241,0.418281,2.147999,-0.034975,-0.024053,0.172504,-0.086361,0.005984,-0.008092,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1497,GAL-QSO-LENS_-0.0431_-0.0218,0.823760,2.148573,-0.043068,-0.021755,0.067924,0.019477,0.000000,0.000000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1498,GAL-QSO-LENS_-0.0228_0.0331,0.310547,2.157330,-0.022781,0.033149,-0.005925,-0.035904,0.034422,0.040130,0.0,...,0.068482,0.075784,0.487043,0.527906,0.548966,0.610543,-0.533254,1.958694,-7.736762,-7.307965
1499,GAL-QSO-LENS_-0.0048_0.0380,0.830419,2.076385,-0.004827,0.037979,-0.053450,0.119525,-0.000057,-0.060998,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1500,GAL-QSO-LENS_0.0042_-0.0128,0.414335,2.080392,0.004218,-0.012772,0.030604,0.047062,-0.000333,0.017460,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Generate images

In [11]:
def get_random_ra_dec(N=1):
    ra_points = coord.Angle(np.random.uniform(low=0, high=360, size=N) * u.degree)
    ra_points = ra_points.wrap_at(180 * u.degree)
    # dec goes from -72 to +12
    lower = -70
    upper = 10
    p = (
        np.sin(np.random.uniform(low=lower, high=upper, size=N) * u.deg)
        - np.sin(lower * u.deg)
    ) / (np.sin(upper * u.deg) - np.sin(lower * u.deg))
    dec_points = coord.Angle(
        ((((np.arcsin(2 * p - 1).to(u.deg) + 90 * u.deg) / (180 * u.deg)) * 84) + lower)
        * u.deg
    )
    return ra_points, dec_points

def compute_magnitude_zeropoint(mag_zp_1s, exposure_time=30, gain=1):
        return mag_zp_1s + 2.5 * np.log10(exposure_time / gain)

In [12]:
lsst_colors = {
    "u": "#0c71ff",
    "g": "#49be61",
    "r": "#c61c00",
    "i": "#ffc200",
    "z": "#f341a2",
    "y": "#5d0000",
}

### Save lens data to HDF5 file

In [22]:
import h5py
def save_lens_to_hdf5(
    filename,
    lens_index,
    metadata_dict,
    observation_dates,
    image_lens_series_all_bands,
    light_curves_dict,
):
    """
    Save lens system data to HDF5 file.
    
    Parameters:
    -----------
    filename : str
        Path to HDF5 file (will be created if doesn't exist)
    lens_index : int
        Index of the lens (e.g., 1 for lsst_lens_1)
    metadata_dict : dict
        Dictionary containing metadata from full_pop_df row
    observation_dates : dict
        Dictionary with keys as bands ('u', 'g', 'r', 'i', 'z', 'y') and values as observation MJD arrays
    image_lens_series_all_bands : list
        List of image arrays for each band (6 bands total)
    light_curves_dict : dict
        Dictionary with structure {image_num: {band: magnitude_array}}
        e.g., {0: {'u': [mag1, mag2, ...], 'g': [mag1, mag2, ...]}, 1: {...}, ...}
    """
    
    # Open file in append mode (creates if doesn't exist)
    with h5py.File(filename, 'a') as hf:
        # Create group for this lens
        lens_group_name = f'lsst_lens_{lens_index}'
        
        # Delete group if it already exists
        if lens_group_name in hf:
            del hf[lens_group_name]
        
        lens_group = hf.create_group(lens_group_name)
        
        # Save metadata
        metadata_group = lens_group.create_group('metadata')
        for key, value in metadata_dict.items():
            # Handle different data types
            if isinstance(value, (int, float, np.integer, np.floating)):
                metadata_group.attrs[key] = value
            elif isinstance(value, str):
                metadata_group.attrs[key] = value
            elif isinstance(value, np.ndarray):
                metadata_group.create_dataset(key, data=value)
            elif value is None or (isinstance(value, float) and np.isnan(value)):
                metadata_group.attrs[key] = 'NaN'
            else:
                # Try to convert to string as fallback
                try:
                    metadata_group.attrs[key] = str(value)
                except:
                    print(f"Warning: Could not save metadata key '{key}' with value {value}")
        
        # Save observation dates for each band
        obs_dates_group = lens_group.create_group('observation_dates')
        bands = ['u', 'g', 'r', 'i', 'z', 'y']
        for band in bands:
            if band in observation_dates:
                obs_dates_group.create_dataset(band, data=np.array(observation_dates[band]))
        
        # Save postage stamp images for each band
        images_group = lens_group.create_group('postage_stamps')
        for i, band in enumerate(bands):
            if i < len(image_lens_series_all_bands):
                band_group = images_group.create_group(band)
                image_series = np.array(image_lens_series_all_bands[i])
                band_group.create_dataset(f'all_observations', data=image_series, compression='gzip')
        
        # Save light curves for each band and image
        lightcurves_group = lens_group.create_group('light_curves')
        # Save light curves for each image and band
        for image_num, bands_lc in light_curves_dict.items():
            image_group = lightcurves_group.create_group(f'image_{image_num}')
            for band, magnitudes in bands_lc.items():
                image_group.create_dataset(band, data=np.array(magnitudes))
    

def generate_and_save_single_lens(lens_index, full_pop_df,baseline=10, filename='lens_finding_postage_stamps.h5'):
    """
    Generate all necessary data for a lens and save it to HDF5.

    Parameters:
    -----------

    lens_index : int
    quasar_lens_population : list
        List of lens class objects
        Path to HDF5 file
    """
    
    # Get metadata from dataframe
    lens_row = full_pop_df.loc[lens_index]
    metadata_dict = lens_row.to_dict()
    # Get lens object and metadata from index
    lens_obj = quasar_lens_population[lens_index]    
    # Get random RA and Dec for observation
    new_ra, new_dec = get_random_ra_dec(N=1)
    lens_obj.ra_image = np.array([new_ra.deg])
    lens_obj.dec_image = np.array([new_dec.deg])
    # # Get a point source coordinate so that you can plot these image center in the plot.
    


    bands = list("ugrizy")
    mag_zps = np.array(
        [
            26.52,
            28.51,
            28.36,
            28.17,
            27.78,
            26.82,
        ]
    )  # taken from https://smtn-002.lsst.io/
    # mag_zero_points_1_second = dict(zip(bands, mag_zps))  # mag

    mag_zero_points_30_seconds = dict(
        zip(bands, compute_magnitude_zeropoint(mag_zps))
    )  # mag
    delta_pix = 0.2  # arcsec/pixel
    num_pix = 33  # pixels
    exp_time = 30  # s
    # pix_coord_list = [
    #     point_source_coordinate_properties(
    #         lens_obj,
    #         band=i,
    #         mag_zero_point=mag_zero_points_30_seconds[i],
    #         delta_pix=delta_pix,
    #         num_pix=num_pix,
    #         transform_pix2angle=np.array([[0.2, 0], [0, 0.2]]),
    #     )
    #     for i in bands
    # ]
    # pix_coord_dict = dict(zip(bands, pix_coord_list))
    
    # Get observation dates
    if baseline==10:
        sql=''
    else:
        sql=f'night < {baseline*365.25}'
    while True:
        try:    
            rubin_df = get_rubin_cadence(new_ra, new_dec, sql=sql)
            break
        except Exception as e:
            new_ra, new_dec = get_random_ra_dec(N=1)
            print(f"get_rubin_cadence failed, retrying: {e}")
    
    observation_dates = rubin_df["observationStartMJD"]
    image_number = lens_obj.image_number
    
    if isinstance(image_number, list):
        image_number = image_number[0]
    # Generate postage stamp images for all bands
    bands = list("ugrizy")
    image_lens_series_all_bands = []
    light_curves_dict = {}
    for img_idx in range(image_number):
        light_curves_dict[img_idx] = {}
    for band in bands:
        try:
            time_sampled = np.array(observation_dates[band])
            repeats = len(time_sampled)
            transform_matrix = np.array([[delta_pix, 0], [0, delta_pix]])
            psf_kernel_list = [None]
            transform_matrix_list = [transform_matrix]
            mag_list = [mag_zero_points_30_seconds[band]]
            expo_list = [exp_time]
            mag_zero_points_all = mag_list * repeats
            psf_kernels_all = psf_kernel_list * repeats
            transform_matrix_all = transform_matrix_list * repeats
            exposure_time_all = expo_list * repeats
            
            image_lens_series, magnitudes = lens_image_series_precomputed_mags(
                lens_class=lens_obj,
                band=band,
                mag_zero_point=mag_zero_points_all,
                num_pix=num_pix,
                psf_kernel=psf_kernels_all,
                transform_pix2angle=transform_matrix_all,
                exposure_time=exposure_time_all,
                std_gaussian_noise=bkg_noise(
                    0.005, 30, np.array(rubin_df.loc[band, "skyBrightness"]), 0.2, 1
                ),
                t_obs=time_sampled,
                with_deflector=True,
                with_ps=True,
                with_source=True,
                add_noise=False,
                single_visit_mag_zero_points=mag_zero_points_30_seconds
            )
            magnitudes = np.array(magnitudes).T

            image_lens_series_all_bands.append(image_lens_series)
            for img_idx in range(image_number):
                light_curves_dict[img_idx][band] = magnitudes[img_idx]
        except KeyError:
            print(f'no observation i {band} band in {baseline} years')
        # print('magnitudes:', magnitudes, magnitudes.shape)
        # magnitudes = lens_obj.point_source_magnitude(
        #     band=band, lensed=True, time=time_sampled, microlensing=True
        # )[0]
        
        

    save_lens_to_hdf5(
        filename=filename,
        lens_index=lens_index, 
        metadata_dict=metadata_dict,
        observation_dates=observation_dates,
        image_lens_series_all_bands=image_lens_series_all_bands,
        light_curves_dict=light_curves_dict)
    
    return lens_obj


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



### Example: Process and save multiple lenses

In [14]:
# generate_and_save_single_lens(0, full_pop_df, 'pls.h5')

### Visualize pls.h5 light curves and postage stamps

In [15]:
# import h5py
# import numpy as np
# import matplotlib.pyplot as plt

# filename = "pls.h5"
# bands = ["u", "g", "r", "i", "z", "y"]

# with h5py.File(filename, "r") as hf:
#     lens_keys = sorted(hf.keys())
#     if not lens_keys:
#         raise ValueError(f"No lens groups found in {filename}")
#     lens_key = lens_keys[0]
#     lens_group = hf[lens_key]
#     lc_group = lens_group["light_curves"]
#     obs_group = lens_group.get("observation_dates", None)
#     image_keys = sorted(lc_group.keys(), key=lambda k: int(k.split("_")[1]))
#     print(f"Using lens group: {lens_key}")

#     # Plot light curves for all images in all bands
#     n_images = len(image_keys)
#     fig, axes = plt.subplots(
#         n_images, 1, figsize=(10, 3 * n_images), sharex=True
#     )
#     if n_images == 1:
#         axes = [axes]
#     for idx, image_key in enumerate(image_keys):
#         ax = axes[idx]
#         for band in bands:
#             if band in lc_group[image_key]:
#                 mags = lc_group[image_key][band][()]
#                 if obs_group is not None and band in obs_group:
#                     times = obs_group[band][()]
#                 else:
#                     times = np.arange(len(mags))
#                 color = lsst_colors.get(band, None)
#                 ax.scatter(times, mags, label=band, color=color)
#         ax.invert_yaxis()
#         ax.set_ylabel(f"{image_key} mag")
#         ax.legend(ncol=3, fontsize=8)
#     axes[-1].set_xlabel("Observation MJD" if obs_group is not None else "Time index")
#     fig.suptitle("Light curves for all images (all bands)")
#     plt.show()

#     # Gather all images to compute global min/max for normalization
#     images_group = lens_group["postage_stamps"]
#     all_images = []
#     for band in bands:
#         if band in images_group:
#             band_group = images_group[band]
#             time_keys = sorted(
#                 band_group.keys(), key=lambda k: int(k.split("_")[1])
#             )
#             for tk in time_keys:
#                 all_images.append(band_group[tk][()])
#     if not all_images:
#         raise ValueError("No images found in postage_stamps group.")
#     global_min = min(img.min() for img in all_images)
#     global_max = max(img.max() for img in all_images)

#     def asinh_stretch(img, vmin, vmax, scale=10.0):
#         img = np.clip(img, vmin, vmax)
#         if vmax > vmin:
#             scaled = (img - vmin) / (vmax - vmin)
#         else:
#             scaled = np.zeros_like(img)
#         return np.arcsinh(scale * scaled) / np.arcsinh(scale)

#     # Plot 10 evenly sampled images for each band with shared normalization + asinh stretch
#     num_samples = 10
#     fig, axes = plt.subplots(
#         len(bands), num_samples, figsize=(2 * num_samples, 2 * len(bands)), constrained_layout=True
#     )
#     for bi, band in enumerate(bands):
#         if band not in images_group:
#             for j in range(num_samples):
#                 axes[bi, j].axis("off")
#             continue
#         band_group = images_group[band]
#         time_keys = sorted(
#             band_group.keys(), key=lambda k: int(k.split("_")[1])
#         )
#         n_times = len(time_keys)
#         if n_times == 0:
#             for j in range(num_samples):
#                 axes[bi, j].axis("off")
#             continue
#         idxs = (
#             np.linspace(0, n_times - 1, num_samples, dtype=int)
#             if n_times >= num_samples
#             else np.arange(n_times)
#         )
#         for j in range(num_samples):
#             ax = axes[bi, j]
#             if j >= len(idxs):
#                 ax.axis("off")
#                 continue
#             tk = time_keys[idxs[j]]
#             img = band_group[tk][()]
#             img_s = asinh_stretch(img, global_min, global_max)
#             ax.imshow(img_s, origin="lower", cmap="gray", vmin=0, vmax=1)
#             if j == 0:
#                 ax.set_ylabel(band, rotation=0, labelpad=15, fontsize=10)
#             ax.set_title(f"t={tk.split('_')[1]}", fontsize=8)
#             ax.axis("off")
#     fig.suptitle("10 evenly sampled postage stamps per band (asinh stretch)")
#     plt.show()

In [17]:
mask = (
    np.array(full_pop_df[[f"micro_kappa_star_{i}" for i in range(4)]])
    > np.array(full_pop_df[[f"micro_kappa_tot_{i}" for i in range(4)]])
).any(axis=1)

full_pop_df = full_pop_df.loc[~mask]
quasar_lens_population = np.array(quasar_lens_population)[~mask]

In [18]:
full_pop_df.to_csv('10000sqdeg_data0212.csv')

### Batch saving

In [ ]:
# Example: Save first 5 lenses (you can change this to 3000)
# Note: This will take some time to run for all 3000 lenses
# lens_index, full_pop_df,baseline=10, filename='lens_finding_postage_stamps.h5'
filename = '/pscratch/sd/v/vpadma/lens_finding/lens_finding_postage_stamps_1_year_v4.h5'
num_lenses_to_save =len(full_pop_df) # Change to 3000 for full dataset

for i in tqdm(range(num_lenses_to_save)):
    ind = full_pop_df.index[i]
    generate_and_save_single_lens(ind, full_pop_df,baseline=1, filename=filename)
        


  0%|                                                                                                                   | 0/1489 [00:00<?, ?it/s]

kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.


  0%|                                                                                                         | 1/1489 [00:04<1:43:46,  4.18s/it]

kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
Generating magnification map ...
Done generating magnification map.
Generating magnification map ...
Done generating magnification map.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.


  0%|▏                                                                                                        | 2/1489 [00:08<1:51:01,  4.48s/it]

...

kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
Generating magnification map ...
Done generating magnification map.
Generating magnification map ...
Done generating magnification map.
Generating magnification map ...
Done generating magnification map.
Generating magnification map ...
Done generating magnification map.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.


 10%|██████████▎                                                                                            | 149/1489 [20:04<2:08:31,  5.75s/it]

kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
Generating magnification map ...
Done generating magnification map.
Generating magnification map ...
Done generating magnification map.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.


 10%|██████████▍                                                                                            | 150/1489 [20:09<2:09:28,  5.80s/it]

kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
Generating magnification map ...
Done generating magnification map.
Generating magnification map ...
Done generating magnification map.
kwargs_magnification_map not in kwargs_microlensing. Using default magnification map kwargs.
